# 08 — Null Model Comparison, Enrichment-on-Nulls, and Positive Control

This notebook implements the central methodological analysis of the manuscript *"Null Model Specification Determines Whether Cross-Lineage Transcriptional Convergence Is Detected: a Worked Example in Neutrophils and NK Cells."*

It reproduces three results reported in Sections 3.2.1, 3.9, and 3.10 of the manuscript:

1. **Independent-shuffle null vs. NK-only (correlation-preserving) null** for the three-way neutrophil/NK convergence test (Table 1, Section 3.2.1).
2. **Enrichment-on-null-sets**: whether the functional enrichment of the 20-gene concordant set (20-GCS) exceeds what a chance, selection-criterion-matched gene set would show (Table 7, Section 3.9).
3. **Positive control**: applying the identical correlation-preserving null to a third, independent neutrophil dataset (GSE288312, IL-1beta) in place of NK cells, including a sensitivity check restricting the null pool to the 8,581-gene shared background (Section 3.10).

Inputs: `tables/ranked_Neut_TNF.rnk`, `tables/ranked_Neut_LPS.rnk`, `tables/ranked_NK_IL2.rnk`, `tables/meta_analysis_RRA_GSE50838_all_genes.csv` (source of the 8,581-gene shared background). GSE288312 fold-changes are re-derived from raw GEO counts (see cell below); the resulting log2FC table is not re-distributed here as it is trivially recomputed from public GEO data (~3 MB gzip).

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

def load_rnk(path):
    d = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            gene, val = parts
            try:
                d[gene] = float(val)
            except ValueError:
                continue
    return d

tnf = load_rnk("../tables/ranked_Neut_TNF.rnk")
lps = load_rnk("../tables/ranked_Neut_LPS.rnk")
nk  = load_rnk("../tables/ranked_NK_IL2.rnk")

bg_df = pd.read_csv("../tables/meta_analysis_RRA_GSE50838_all_genes.csv")
background = set(bg_df["Gene"])
print(f"Background (8,581-gene shared background): {len(background)} genes")
print(f"TNF: {len(tnf)} genes | LPS: {len(lps)} genes | NK: {len(nk)} genes")

## 1. Independent-shuffle null vs. NK-only (correlation-preserving) null

Both nulls test the same observed count: genes with |log2FC| > 1, concordant direction, in TNF-alpha, LPS, and NK simultaneously (the 20-GCS).

- **Independent-shuffle null**: shuffles the TNF, LPS, and NK fold-change vectors independently across the 8,581-gene background, discarding the real TNF/LPS correlation.
- **NK-only null**: holds the real (TNF, LPS) pairing fixed for every gene and permutes only which gene's fold-change gets assigned to the NK slot — the hypothesis-appropriate design, since it does not pretend TNF and LPS are independent when they are not.

In [ ]:
genes = sorted(background & set(tnf) & set(lps) & set(nk))
tnf_v = np.array([tnf[g] for g in genes])
lps_v = np.array([lps[g] for g in genes])
nk_v  = np.array([nk[g] for g in genes])
N = len(genes)
print(f"Genes with values in all three datasets and the shared background: {N}")

def concordant_count(t, l, n, thresh=1.0):
    up = (t > thresh) & (l > thresh) & (n > thresh)
    down = (t < -thresh) & (l < -thresh) & (n < -thresh)
    return int(np.sum(up | down))

observed_20gcs = concordant_count(tnf_v, lps_v, nk_v)
print(f"Observed 3-way concordant genes (20-GCS): {observed_20gcs}")

ITERS = 10000

# Independent-shuffle null: permute all three vectors independently
null_indep = np.empty(ITERS, dtype=int)
for i in range(ITERS):
    t_p = np.random.permutation(tnf_v)
    l_p = np.random.permutation(lps_v)
    n_p = np.random.permutation(nk_v)
    null_indep[i] = concordant_count(t_p, l_p, n_p)

# NK-only null: TNF/LPS pairing held fixed, only NK permuted
null_nkonly = np.empty(ITERS, dtype=int)
for i in range(ITERS):
    n_p = np.random.permutation(nk_v)
    null_nkonly[i] = concordant_count(tnf_v, lps_v, n_p)

print("\nIndependent-shuffle null: mean={:.2f} sd={:.2f} max={} | empirical p={:.4f}".format(
    null_indep.mean(), null_indep.std(), null_indep.max(),
    (null_indep >= observed_20gcs).mean()))
print("NK-only null:             mean={:.2f} sd={:.2f} max={} | empirical p={:.4f}".format(
    null_nkonly.mean(), null_nkonly.std(), null_nkonly.max(),
    (null_nkonly >= observed_20gcs).mean()))

Expected output (manuscript, Section 3.2.1): independent-shuffle null mean 4.8 +/- 2.2 (p < 0.0001); NK-only null mean 20.7 +/- 4.3 (p = 0.60) against an observed count of 20. Exact null statistics vary slightly run-to-run due to Monte Carlo sampling; the qualitative conclusion (p < 0.0001 vs. p = 0.60) is stable.

## 2. Enrichment-on-null-sets (Table 7)

For each of the 21 enrichment terms reported in Tables 2 and 4 (ChEA 2022 / GO Biological Process 2021 / KEGG 2021 Human, retrieved via the Enrichr `geneSetLibrary` endpoint), this asks whether the term's overlap with the *observed* 20-GCS exceeds what a chance, NK-only-permuted 20-gene set would show. The gene-set libraries themselves (large term -> gene-list mappings from Enrichr) are not redistributed in this repository; this cell illustrates the logic using a placeholder library dict that should be replaced with the real Enrichr gene sets (see `tables/ChEA2022_TF_enrichment_20GCS.csv`, `tables/GO_enrichment_20GCS.csv`, `tables/KEGG_enrichment_20GCS.csv` for the term lists actually used, and `tables/enrichment_on_null_sets.csv` for the precomputed result reported as Table 7).

In [ ]:
def null_gcs_gene_sets(tnf_v, lps_v, nk_v, genes, n_iter=10000, thresh=1.0):
    """Yield the gene set corresponding to each NK-only null permutation."""
    genes = np.array(genes)
    for _ in range(n_iter):
        idx = np.random.permutation(len(nk_v))
        n_p = nk_v[idx]
        up = (tnf_v > thresh) & (lps_v > thresh) & (n_p > thresh)
        down = (tnf_v < -thresh) & (lps_v < -thresh) & (n_p < -thresh)
        yield set(genes[up | down])

def enrichment_on_nulls(term_gene_list, observed_gcs, tnf_v, lps_v, nk_v, genes, n_iter=10000):
    term_set = set(term_gene_list)
    obs_k = len(observed_gcs & term_set)
    null_overlaps = np.array([
        len(ns & term_set) for ns in null_gcs_gene_sets(tnf_v, lps_v, nk_v, genes, n_iter)
    ])
    p = (null_overlaps >= obs_k).mean()
    return obs_k, null_overlaps.mean(), null_overlaps.std(), p

# Example: RELA (ChEA) — reported in the manuscript as the weakest-supported term (p = 0.78)
# rela_targets = load_enrichr_geneset("ChEA_2022", "RELA")  # requires Enrichr geneSetLibrary fetch
# print(enrichment_on_nulls(rela_targets, observed_gcs=set(genes)[...], ...))

print("See tables/enrichment_on_null_sets.csv for the full precomputed 21-term result (Table 7).")

## 3. Positive control: GSE288312 (IL-1beta neutrophils)

Tests whether the pipeline can detect convergence when it genuinely exists, using a case where biological convergence is expected: three independent neutrophil pro-inflammatory stimuli (TNF-alpha, LPS, IL-1beta) instead of two neutrophil datasets plus an unrelated lymphoid lineage.

GSE288312 raw counts are downloaded directly from GEO and converted to a simplified CPM-based log2 fold-change: `log2((mean IL-1beta CPM + 1) / (mean untreated CPM + 1))`. This is disclosed in the manuscript as a simplified estimate (not a full DESeq2/edgeR model), appropriate for this supplementary check.

In [ ]:
# --- Step 1: build the 224-gene TNF-alpha/LPS two-way concordant candidate set ---
# (restricted to the 8,581-gene shared background — see Section 3.2.1 of the manuscript)
common = set(tnf) & set(lps) & background
candidates = []
for g in common:
    t, l = tnf[g], lps[g]
    if abs(t) > 1 and abs(l) > 1 and (t > 0) == (l > 0):
        candidates.append((g, 1 if t > 0 else -1))

n_up = sum(1 for _, d in candidates if d == 1)
n_down = sum(1 for _, d in candidates if d == -1)
print(f"224-gene candidate set: {len(candidates)} genes ({n_up} up, {n_down} down)")
assert len(candidates) == 224, "Expected 224 candidates; see tables/candidates_224_TNF_LPS_concordant.csv"

In [ ]:
# --- Step 2: load GSE288312 (IL-1beta) log2FC. Re-derive from raw GEO counts: ---
#
#   import gzip, urllib.request, io
#   url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE288nnn/GSE288312/suppl/GSE288312_raw_counts.csv.gz"
#   raw = urllib.request.urlopen(url).read()
#   counts = pd.read_csv(io.BytesIO(gzip.decompress(raw)), index_col=0)
#   cpm = counts.div(counts.sum(axis=0), axis=1) * 1e6
#   il1b_samples = [...]      # IL-1beta-treated columns (n=3)
#   untreated_samples = [...] # untreated columns (n=6)
#   il1b_fc = np.log2((cpm[il1b_samples].mean(axis=1) + 1) / (cpm[untreated_samples].mean(axis=1) + 1))
#
# For this notebook, load a precomputed log2FC table if available:
try:
    il1b_df = pd.read_csv("../tables/GSE288312_IL1B_log2FC.csv", index_col=0)
    il1b_fc = il1b_df.iloc[:, 0]
    HAVE_IL1B = True
except FileNotFoundError:
    print("GSE288312_IL1B_log2FC.csv not found locally — re-derive from raw GEO counts (see comment above) to run this section.")
    HAVE_IL1B = False

In [ ]:
def positive_control_test(candidates, il1b_fc, pool_genes, n_iter=10000, thresh=1.0):
    """Correlation-preserving positive-control null: only the IL-1beta assignment is
    permuted across pool_genes; the real TNF-alpha/LPS pairing (candidate direction) is held fixed.
    """
    pool_vals = np.array([il1b_fc[g] for g in pool_genes if g in il1b_fc.index])
    dirs = np.array([d for g, d in candidates if g in il1b_fc.index])
    fc = np.array([il1b_fc[g] for g, d in candidates if g in il1b_fc.index])

    observed = int(np.sum((np.abs(fc) > thresh) & (np.sign(fc) == dirs)))

    K = len(dirs)
    null_counts = np.empty(n_iter, dtype=int)
    for i in range(n_iter):
        draw = np.random.choice(pool_vals, size=K, replace=False)
        null_counts[i] = int(np.sum((np.abs(draw) > thresh) & (np.sign(draw) == dirs)))

    p = (null_counts >= observed).mean()
    return {
        "n_pool": len(pool_vals), "n_candidates": K, "observed": observed,
        "null_mean": null_counts.mean(), "null_sd": null_counts.std(),
        "null_max": int(null_counts.max()), "empirical_p": p,
    }

if HAVE_IL1B:
    # Primary analysis: unrestricted background (all genes profiled in GSE288312)
    pool_all = list(il1b_fc.index)
    res_unrestricted = positive_control_test(candidates, il1b_fc, pool_all)
    print("Unrestricted background (all GSE288312-profiled genes):", res_unrestricted)

    # Sensitivity check: restrict the null pool to the 8,581-gene shared background
    pool_restricted = [g for g in background if g in il1b_fc.index]
    res_restricted = positive_control_test(candidates, il1b_fc, pool_restricted)
    print("Restricted to 8,581-gene shared background:", res_restricted)

Expected output (manuscript, Section 3.10): unrestricted background — 12/224 concordant, null mean 0.39 +/- 0.62, max 4, p < 0.0001. Restricted 8,581-gene background (sensitivity check) — same 12/224 observed (all 224 candidates and all 8,581 background genes are profiled in GSE288312), null mean 0.88 +/- 0.92, max 7, p < 0.0001. See `tables/positive_control_GSE288312.csv` for the precomputed result of both variants.